# Hex6 Colab Training

This notebook can run either a one-shot bootstrap training job or a longer Colab cycle session.
Use `configs/fast.toml` only when you want the quickest possible smoke test.

If `HEX6_GITHUB_TOKEN` is set in Colab secrets, the run will publish live status JSON
to the `colab-status` branch so the local watcher can detect completion.

In [ ]:
REPO_MODE = "git"  # set to "drive" to use a repo folder in Google Drive
REPO_URL = "https://github.com/Stroudmj00/hex6-bot.git"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Hexagonal tic tac toe"
WORKDIR = "/content/hex6-bot"
RUN_MODE = "cycle"  # "bootstrap" or "cycle"
CONFIG_PATH = "configs/colab_hour.toml"
OUTPUT_DIR = "artifacts/bootstrap_colab_hour"
CYCLE_MINUTES = 60
ARTIFACT_TARGET = "/content/drive/MyDrive/hex6_artifacts/bootstrap_colab_hour"


In [ ]:
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

RUN_ID = f"colab-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
print("RUN_ID:", RUN_ID)

try:
    from google.colab import userdata
    token = userdata.get("HEX6_GITHUB_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HEX6_GITHUB_TOKEN"] = token
    print("Loaded HEX6_GITHUB_TOKEN from Colab secrets.")
else:
    print("No HEX6_GITHUB_TOKEN secret found. GitHub status publishing will fail if enabled.")

if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    shutil.copytree(DRIVE_REPO_PATH, WORKDIR)
else:
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("WORKDIR:", os.getcwd())
print("Local watch command:")
print(f".\\.venv\\Scripts\\python -m hex6.integration.watch_status --config {CONFIG_PATH} --run-id {RUN_ID}")


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "numpy"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run(
    [
        "python",
        "-c",
        "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')",
    ],
    check=True,
)


In [ ]:
if RUN_MODE == "cycle":
    command = [
        "python",
        "-m",
        "hex6.train.run_cycle",
        "--config",
        CONFIG_PATH,
        "--output-root",
        OUTPUT_DIR,
        "--minutes",
        str(CYCLE_MINUTES),
        "--run-id",
        RUN_ID,
    ]
else:
    command = [
        "python",
        "-m",
        "hex6.train.run_bootstrap",
        "--config",
        CONFIG_PATH,
        "--output",
        OUTPUT_DIR,
        "--run-id",
        RUN_ID,
    ]
if not token:
    command.extend(["--status-backend", "none"])
subprocess.run(command, check=True)


In [ ]:
result_path = Path(OUTPUT_DIR) / ("cycle_summary.json" if RUN_MODE == "cycle" else "metrics.json")
print(result_path.read_text())
print("Status file:", "https://raw.githubusercontent.com/Stroudmj00/hex6-bot/colab-status/status/latest.json")

if REPO_MODE == "drive":
    target = Path(ARTIFACT_TARGET)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(OUTPUT_DIR, target)
    print("Artifacts copied to", target)
